# СӨЖ · 13-апта: CNN архитектуралары

Ақылбек, Алмас және Нұрдаулет

Архитектура — нейрондық желінің құрылыс жоспары: қандай қабат бар, қанша рет қайталанады және қалай байланысады. Бір CNN-нің салмағын өзгертсек, оның үйренгені өзгереді; қабаттар мен байланыстарды өзгертсек, архитектурасы өзгереді.

Conv белгілерді есептейді. Pooling картаны кішірейтеді. Dense белгілерді біріктіреді. Канал — бір белгінің картасы. H×W×C жазуы биіктікті, енді және канал санын білдіреді.

Бұл бөлім құрылымдарды түсіндіреді. Анимация — деректің қабаттар арқылы өтуінің сызбасы; оқытылған модельдің дәлдігі емес. Python мысалдары құрылып, бір жасанды кіріспен тексеріледі.

Бұл notebook құрылымдарды зерттейді. Оқыту орындалмайды. Random weights → synthetic forward pass.


In [ ]:
%pip install tensorflow numpy


## Орнату және импорттар

Алдымен pip install tensorflow numpy. Сосын python cnn-architectures.py --model lenet. Басқа таңдау: alexnet, vgg16, inception, residual, resnet50. Әр іске қосуда бір модель ғана құрылады.

weights=None — салмақтар кездейсоқ бастапқы мәнмен беріледі, дайын білім жүктелмейді. Жасанды кірістегі forward pass тек өлшемдердің үйлесуін тексереді; оның жауабы суретті тану нәтижесі емес. Оқыту мен test accuracy бұл СӨЖ-де орындалды деп көрсетілмейді.

LeNet және AlexNet — анық белгіленген оқу нұсқалары; inception және residual — жеке блоктар. VGG16 мен ResNet50 — Keras модельдерінің классификаторсыз нұсқалары. Notebook-та ұяшықтарды жоғарыдан төмен орында.


In [1]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TF_NUM_INTRAOP_THREADS", "2")
os.environ.setdefault("TF_NUM_INTEROP_THREADS", "2")

import argparse
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(13)


## LeNet: оқу нұсқасы

32×32 бір арналы сурет 5×5 сүзгіден өткенде 28×28 болады: 32−5+1=28. Алты сүзгі алты карта шығарады. Кішірейту қабаты оларды 14×14-ке түсіреді. Кейін 16 карта, 120 белгі, 84 белгі және 10 жауапқа өтеміз.

Түпнұсқа LeNet-5-те үйренілетін subsampling, кейбір каналдар арасында ғана байланыс және RBF шығыс бар. RBF — класс үлгілеріне ұқсастықты қашықтық арқылы бағалайтын шығыс. Жүктелетін оқу нұсқасы AveragePooling, толық байланысқан Conv және softmax қолданады; ол түпнұсқаның дәл көшірмесі емес.


In [2]:
def make_lenet():
    """Teaching LeNet variant: full channel connections, fixed average pooling,
    tanh activations, softmax output. The original used trainable subsampling,
    partial C3 connectivity, and an RBF output."""
    return keras.Sequential([
        layers.Input((32, 32, 1)),
        layers.Conv2D(6, 5, activation="tanh", name="C1"),
        layers.AveragePooling2D(2, name="S2_average"),
        layers.Conv2D(16, 5, activation="tanh", name="C3_full_connections"),
        layers.AveragePooling2D(2, name="S4_average"),
        layers.Conv2D(120, 5, activation="tanh", name="C5"),
        layers.Flatten(),
        layers.Dense(84, activation="tanh", name="F6"),
        layers.Dense(10, activation="softmax", name="teaching_softmax"),
    ], name="LeNet_teaching_variant")


## AlexNet: оқу нұсқасы

AlexNet-те 5 convolution және 3 толық байланысқан қабат бар. Соңғы 1000 шығыс ImageNet кластарына сәйкес келеді. RGB суретте үш түс каналы бар. Алғашқы үлкен 11×11 сүзгі 4 қадаммен қозғалады. Сызбадағы 227×227 — 55×55 нәтиже беретін кең таралған іске асыру нұсқасы.

ReLU теріс жауапты нөлге ауыстырады. Dropout оқытуда кейбір белсендіруді уақытша нөлге түсіріп, шамадан тыс жаттауды азайтуға көмектеседі. Біздің кодта екі GPU-ға бөлу мен LRN жоқ. LRN — көршілес каналдар жауаптарын нормалау амалы. Сондықтан код оқу нұсқасы деп белгіленген.


In [3]:
def make_alexnet():
    """Teaching AlexNet variant with the common 227x227 input convention.
    Full channel connections: no original two-GPU grouping or LRN.
    Dropout affects training only; the demo calls training=False."""
    return keras.Sequential([
        layers.Input((227, 227, 3)),
        layers.Conv2D(96, 11, strides=4, activation="relu"),
        layers.MaxPooling2D(3, strides=2),
        layers.Conv2D(256, 5, padding="same", activation="relu"),
        layers.MaxPooling2D(3, strides=2),
        layers.Conv2D(384, 3, padding="same", activation="relu"),
        layers.Conv2D(384, 3, padding="same", activation="relu"),
        layers.Conv2D(256, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(3, strides=2),
        layers.Flatten(),
        layers.Dense(4096, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(4096, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(1000, activation="softmax"),
    ], name="AlexNet_teaching_variant")


## VGG16: классификаторсыз

VGG16 атауындағы 16 — 13 convolution және 3 Dense қабатының саны. Pooling қабаттары бұл санға кірмейді. 3×3 convolution-дар блок ішінде қайталанады; pooling блоктар арасында өлшемді екі есе кішірейтеді.

Бірдей C канал, bias жоқ деп алсақ: бір 7×7 convolution 49C² салмақ, ал үш 3×3 convolution 27C² салмақ қажет етеді. Үш 3×3 қабаттың қамтитын кіріс аумағы 7×7, бірақ араларындағы ReLU арқасында есептері бірдей емес.

Сызба 224×224 RGB кірісі және классификаторы бар толық VGG16-ны көрсетеді. Python мысалы жеңіл тексеру үшін 64×64 RGB қолданады: include_top=False соңғы 3 Dense қабатын алып тастайды, pooling="avg" әр картадағы мәндерді орташа біріктіріп, 512 белгі шығарады. weights=None — кездейсоқ бастапқы салмақтар; бұл үйретілген жіктеу емес.


In [4]:
def make_vgg16():
    # Original complete VGG16 has 13 Conv + 3 Dense layers.
    # include_top=False removes those 3 Dense layers.
    # 64x64 is used for a lightweight structural demonstration, not ImageNet.
    # Global average pooling produces 512 features, NOT class probabilities.
    return keras.applications.VGG16(
        weights=None, include_top=False, input_shape=(64, 64, 3), pooling="avg"
    )


## Inception v1: бір 3a блогы

GoogLeNet 2014 жылы таныстырылды, мақаласы CVPR 2015-те шықты. Оның Inception блогында 1×1, 3×3, 5×5 сүзгілер және pooling тармағы қатар жұмыс істейді. Нәтижелер канал осі бойымен тіркеледі: бұл қосу емес.

Көрсетілген 3a блогы 28×28×192 кірістен 64, 128, 32 және 32 канал шығарады: барлығы 256. Барлық тармақта биіктік пен ен бірдей болуы керек. 1×1 сүзгі қымбат 3×3 не 5×5 амалдың алдында канал санын азайтады.

Код толық GoogLeNet-ті емес, осы бір Inception v1 блогын құрады. InceptionV3 — басқа, кейінгі архитектура; оны бұл жерде GoogLeNet деп атамаймыз.


In [5]:
def inception_v1_3a(x):
    """One original-style Inception-v1 3a block.
    All four outputs preserve H and W, then concatenate their channels."""
    branch1 = layers.Conv2D(64, 1, activation="relu", name="branch1_1x1")(x)

    branch3 = layers.Conv2D(96, 1, activation="relu", name="branch3_reduce")(x)
    branch3 = layers.Conv2D(128, 3, padding="same", activation="relu",
                           name="branch3_3x3")(branch3)

    branch5 = layers.Conv2D(16, 1, activation="relu", name="branch5_reduce")(x)
    branch5 = layers.Conv2D(32, 5, padding="same", activation="relu",
                           name="branch5_5x5")(branch5)

    branch_pool = layers.MaxPooling2D(3, strides=1, padding="same",
                                     name="branch_pool")(x)
    branch_pool = layers.Conv2D(32, 1, activation="relu",
                               name="branch_pool_projection")(branch_pool)
    # 64 + 128 + 32 + 32 = 256 output channels.
    return layers.Concatenate(axis=-1, name="concatenate_channels")(
        [branch1, branch3, branch5, branch_pool]
    )


def make_inception():
    x = layers.Input((28, 28, 192))
    return keras.Model(x, inception_v1_3a(x), name="Inception_v1_3a_block")


## Residual блок және ResNet50

ResNet-тің негізгі идеясы: y=F(x)+x. x — блоктың кірісі, F(x) — convolution қабаттары есептеген өзгеріс. Екі жолдағы сәйкес сандар қосылады. Мысалы, x=4 және F(x)=−1 болса, қосынды 3.

Мұндай байланыс өте терең желілерді оқытуды жеңілдетеді. Бірақ ол кез келген деректе міндетті түрде ең жоғары дәлдік береді деген сөз емес. Қосу үшін екі жолдың өлшемдері сәйкес болуы керек; сәйкес болмаса 1×1 projection қолданылады.

ResNet50 сызбасында 1×1→3×3→1×1 bottleneck блоктары бар. Bottleneck ішкі канал санын азайтып, кейін қайта кеңейтеді. Сандық анимация бір қосу амалын түсіндіреді. Осы беттің Python кодында екі бөлек нұсқа бар: --model residual екі 3×3 қабаты бар шағын оқу блогын, --model resnet50 Keras ResNet50 моделін іске қосады.

Толық ResNet50 сызбасы 224×224 RGB кірісін және 1000 класты шығысты көрсетеді. Python-дағы Keras нұсқасы 64×64 RGB қолданады, include_top=False классификаторды алып тастайды, pooling="avg" 2048 белгі береді. weights=None кездейсоқ салмақтарды білдіреді; бұл 2048 сан класс ықтималдықтары емес.


In [6]:
def residual_block(x, filters, stride=1):
    """Educational ResNet-v1 basic block: two 3x3 convolutions.
    Full ResNet50 instead uses 1x1 -> 3x3 -> 1x1 bottleneck blocks."""
    shortcut = x
    y = layers.Conv2D(filters, 3, strides=stride, padding="same",
                      use_bias=False)(x)
    y = layers.BatchNormalization()(y)
    y = layers.Activation("relu")(y)
    y = layers.Conv2D(filters, 3, padding="same", use_bias=False)(y)
    y = layers.BatchNormalization()(y)

    # Add requires matching H, W and C on both paths.
    if stride != 1 or x.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride,
                                 use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    y = layers.Add()([y, shortcut])
    return layers.Activation("relu")(y)


def make_residual():
    x = layers.Input((32, 32, 16))
    return keras.Model(x, residual_block(x, 16), name="Residual_basic_block")


def make_resnet50():
    # Keras ResNet50 without its final classifier; 2048 output features.
    return keras.applications.ResNet50(
        weights=None, include_top=False, input_shape=(64, 64, 3), pooling="avg"
    )


## Құрылымды тексеру

LeNet — шағын тізбекті CNN. AlexNet — үлкен суреттерге арналған тереңірек CNN. VGG — бірдей шағын сүзгілердің тізбегі. GoogLeNet — параллель тармақтар. ResNet — кірісті қосатын қысқа байланыстар.

Қабат саны мен параметр саны бірдей ұғым емес. Параметр — үйренілетін сан. Үлкен Dense қабаты бірнеше шағын convolution қабатынан көбірек параметр алуы мүмкін. Сондықтан «тереңірек» әрдайым «көбірек салмақ» дегенді білдірмейді.

Жылдар мен тарихи тереңдік мақаладағы модельге қатысты. Python-дағы қысқартылған нұсқаларда қабат және параметр саны басқаша болуы мүмкін. Дәлдікті тек құрылым сызбасынан анықтай алмаймыз.


In [7]:
BUILDERS = {
    "lenet": make_lenet,
    "alexnet": make_alexnet,
    "vgg16": make_vgg16,
    "inception": make_inception,
    "residual": make_residual,
    "resnet50": make_resnet50,
}


def inspect_model(name="lenet", verbose=True):
    """Construct ONE model and run a synthetic input through it.
    Do not interpret random outputs as learned predictions."""
    keras.backend.clear_session()
    keras.utils.set_random_seed(13)
    model = BUILDERS[name]()
    if verbose:
        model.summary()

    input_shape = tuple(int(d) for d in model.input_shape[1:])
    # The leading 1 is batch size: one synthetic example.
    example = np.random.default_rng(13).normal(size=(1,) + input_shape).astype("float32")
    output = model(example, training=False).numpy()
    assert np.isfinite(output).all(), "Non-finite forward output"
    if name in ("lenet", "alexnet"):
        np.testing.assert_allclose(output.sum(axis=1), 1.0, atol=1e-5)
    result = {
        "model": name,
        "input_shape": list(example.shape),
        "output_shape": list(output.shape),
        "parameters": model.count_params(),
        "status": "Random weights; synthetic forward pass only; not trained",
    }
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return result




## Бір модельді іске қос

MODEL мәні: lenet, alexnet, vgg16, inception, residual немесе resnet50. Бір іске қосуда бір модель құрылады. AlexNet үлкен Dense қабаттары үшін көбірек жад пайдаланады.

Нәтижедегі input_shape/output_shape — кіріс/шығыс пішіні, parameters — параметр саны. Шығыс сандары кездейсоқ салмақтардан есептеледі; тану дәлдігі емес.


In [8]:
MODEL = "lenet"
result = inspect_model(MODEL)


Model: "LeNet_teaching_variant"
┌─────────────────────────────────┬────────────────────────┬───────────────┐
│ Layer (type)                    │ Output Shape           │       Param # │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ C1 (Conv2D)                     │ (None, 28, 28, 6)      │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ S2_average (AveragePooling2D)   │ (None, 14, 14, 6)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ C3_full_connections (Conv2D)    │ (None, 10, 10, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ S4_average (AveragePooling2D)   │ (None, 5, 5, 16)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ C5 (Conv2D)                     │ (None, 1, 1, 120)      │        48,120 │
├─────────────────────────────────┼─────────

## Дереккөздер

- [LeCun et al. · Gradient-Based Learning Applied to Document Recognition (1998)](https://princetonuniversity.github.io/NEU-PSY-502/_static/pdf/Class%204/LeCunn1998.pdf)
- [Krizhevsky et al. · AlexNet (2012)](https://papers.nips.cc/paper_files/paper/2012/hash/c399862d3b9d6b76c8436e924a68c45b-Abstract.html)
- [Simonyan & Zisserman · VGG (2014)](https://arxiv.org/abs/1409.1556)
- [Szegedy et al. · Going Deeper with Convolutions (2015)](https://openaccess.thecvf.com/content_cvpr_2015/html/Szegedy_Going_Deeper_With_2015_CVPR_paper.html)
- [He et al. · Deep Residual Learning (2016)](https://openaccess.thecvf.com/content_cvpr_2016/papers/He_Deep_Residual_Learning_CVPR_2016_paper.pdf)
- [Keras · VGG16 / VGG19](https://keras.io/api/applications/vgg/)
- [Keras · ResNet models](https://keras.io/api/applications/resnet/resnet_models/)
